# Practice 6 – ECG signal quality assessment
Biomedical Signal Processing Laboratory

Run the setup cells in order. **Exercise 1 uses only set A**: load set B only when you reach Exercise 2.

In [ ]:
!git clone https://github.com/BSP-laboratory/practice_6.git

In [ ]:
!pip install neurokit2==0.2.13

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal as signal
import scipy.stats as stats
import neurokit2 as nk
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score

## Data
`practice_6/sqi_dataset/` contains `set_A.npz` and `set_B.npz`. Each file contains:

| Key | Content |
|---|---|
| `ecg` | windows × 2500: 10-s ECG windows in mV at 250 Hz (one row = one window) |
| `acc` | windows × 250: acceleration magnitude in g at 25 Hz, same windows |
| `label` | quality class of each window: 1, 2 or 3 |
| `fs_ecg`, `fs_acc` | sampling frequencies (Hz) |

In [ ]:
DATA_DIR = "practice_6/sqi_dataset"

def load_set(name):
    """Load set "A" or "B". Returns ecg, acc, label, fs_ecg, fs_acc."""
    d = np.load(f"{DATA_DIR}/set_{name}.npz")
    return d["ecg"], d["acc"], d["label"], int(d["fs_ecg"]), int(d["fs_acc"])

ecg_A, acc_A, label_A, fs_ecg, fs_acc = load_set("A")

print(f"ECG: {ecg_A.shape[0]} windows x {ecg_A.shape[1]} samples, fs = {fs_ecg} Hz")
print(f"ACC: {acc_A.shape[0]} windows x {acc_A.shape[1]} samples, fs = {fs_acc} Hz")
print("Windows per class:", {int(c): int(n) for c, n in zip(*np.unique(label_A, return_counts=True))})

## Helper functions

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title=""):
    """Confusion matrix with counts and row percentages (recall of each true class)."""
    classes = [1, 2, 3]
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    ax.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3):
            pct = 100 * cm[i, j] / max(cm[i].sum(), 1)
            ax.text(j, i, f"{cm[i, j]}\n{pct:.0f}%", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(3), classes)
    ax.set_yticks(range(3), classes)
    ax.set_xlabel("predicted class")
    ax.set_ylabel("true class")
    ax.set_title(f"{title}\naccuracy = {accuracy_score(y_true, y_pred):.3f}")
    plt.tight_layout()
    plt.show()

def plot_window(ecg, acc, label, i):
    """ECG and acceleration magnitude of window i."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 3.2), sharex=True,
                                   gridspec_kw=dict(height_ratios=[3, 1]))
    ax1.plot(np.arange(ecg.shape[1]) / fs_ecg, ecg[i], color="k", lw=0.6)
    ax1.set_ylabel("ECG (mV)")
    ax1.set_title(f"window {i} – class {label[i]}", loc="left")
    ax2.plot(np.arange(acc.shape[1]) / fs_acc, acc[i], color="0.4", lw=0.8)
    ax2.set_ylabel("|acc| (g)")
    ax2.set_xlabel("time (s)")
    plt.tight_layout()
    plt.show()

plot_window(ecg_A, acc_A, label_A, 0)

## Example: R-peak detection with NeuroKit
`nk.ecg_clean` filters the ECG (high-pass 0.5 Hz + powerline), `nk.ecg_peaks` returns the R-peak positions
as sample indices. Divide by `fs_ecg` to get times in seconds.

In [ ]:
i = 921                                  # window to analyse (a clean class-1 window)
x = ecg_A[i]

x_clean = nk.ecg_clean(x, sampling_rate=fs_ecg)
_, info = nk.ecg_peaks(x_clean, sampling_rate=fs_ecg)
r_peaks = info["ECG_R_Peaks"]            # sample indices of the R peaks

rr = np.diff(r_peaks) / fs_ecg           # RR intervals (s)
print(f"{len(r_peaks)} R peaks detected")
print("RR intervals (s):", np.round(rr, 2))

t = np.arange(len(x)) / fs_ecg
plt.figure(figsize=(12, 3))
plt.plot(t, x, color="k", lw=0.6)
plt.plot(r_peaks / fs_ecg, x[r_peaks], "o", mfc="none", mec="tab:red")
plt.xlabel("time (s)")
plt.ylabel("ECG (mV)")
plt.title(f"window {i} – class {label_A[i]} – circles: detected R peaks", loc="left")
plt.tight_layout()
plt.show()

## Reference values
Use them to check your SQI implementation (windows 0–3 of set A, computed with neurokit2 0.2.13).
cSQI = 1 means that fewer than 3 R peaks were detected (convention, see the slides).

| Window | Class | kSQI | sSQI | pSQI | basSQI | cSQI | R peaks |
|---|---|---|---|---|---|---|---|
| 0 | 1 | 23.7182 | 4.0713 | 0.6349 | 0.8659 | 0.3817 | 17 |
| 1 | 3 | 28.0321 | -3.1421 | 0.8333 | 0.5303 | 0.4060 | 11 |
| 2 | 3 | 5.6701 | 1.0320 | 0.0529 | 0.9998 | 1.0000 | 0 |
| 3 | 1 | 26.6261 | 4.3162 | 0.6456 | 0.8819 | 0.3478 | 17 |

## Exercise 1
Use only set A.

## Exercise 2

In [ ]:
ecg_B, acc_B, label_B, _, _ = load_set("B")

## Exercise 3